# CNN for MNIST Digit Classification: PyTorch Guide

This notebook demonstrates building, training, and evaluating a Convolutional Neural Network (CNN) on the MNIST dataset using PyTorch. The architecture is equivalent to the TensorFlow version, enabling direct framework comparison.

**Learning Objectives:**
- Load and preprocess MNIST data with PyTorch DataLoaders
- Build a CNN using torch.nn.Module
- Implement a custom training loop with backward pass and optimization
- Evaluate performance and compare with TensorFlow
- Log experiments with MLflow for reproducibility

## 1. Setup & Imports

Import PyTorch, torchvision for data loading, and supporting libraries for training and visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import mlflow
import time

# Determine device (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

## 2. Load MNIST Data

Use torchvision to download and load MNIST. PyTorch uses DataLoaders for efficient batching. Note: PyTorch uses NCHW format (channels first) unlike TensorFlow's NHWC.

In [ ]:
# Define transform: convert images to tensors and normalize to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()  # Automatically converts to [0, 1] and format (C, H, W)
])

# Load training and test datasets
train_full = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Split training data into train (50k) and validation (10k)
train_dataset, val_dataset = torch.utils.data.random_split(
    train_full, [50000, 10000], generator=torch.Generator().manual_seed(42)
)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Create DataLoaders for batch processing
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nDataLoader info:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 3. Data Visualization

Display sample images from the training set. Note the tensor format: (1, 28, 28) = (channels, height, width).

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for i in range(10):
    image, label = train_dataset[i]
    axes[i].imshow(image.squeeze(), cmap='gray')  # Remove channel dimension for display
    axes[i].set_title(f"Label: {label}")
    axes[i].axis('off')
plt.suptitle("Sample MNIST Images (PyTorch)")
plt.tight_layout()
plt.show()

# Verify tensor shapes
sample_image, sample_label = train_dataset[0]
print(f"Sample image shape: {sample_image.shape}  (channels, height, width)")
print(f"Sample label: {sample_label}")

## 4. Model Definition

Build a CNN using torch.nn.Module. PyTorch requires explicit forward() method. The architecture mirrors the TensorFlow version for fair comparison.

In [ ]:
class MNISTCNNModel(nn.Module):
    """CNN for MNIST digit classification."""
    
    def __init__(self, seed=42):
        super().__init__()
        torch.manual_seed(seed)
        
        # Conv Block 1: 1 → 32 filters
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)  # 28×28 → 14×14
        
        # Conv Block 2: 32 → 64 filters
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)  # 14×14 → 7×7
        
        # Dense layers
        self.flatten_size = 64 * 7 * 7  # 3136 features
        self.fc1 = nn.Linear(self.flatten_size, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        """Forward pass through CNN."""
        # Conv Block 1
        x = self.conv1(x)
        x = torch.nn.functional.relu(x)
        x = self.pool1(x)
        
        # Conv Block 2
        x = self.conv2(x)
        x = torch.nn.functional.relu(x)
        x = self.pool2(x)
        
        # Dense layers
        x = x.flatten(start_dim=1)
        x = self.fc1(x)
        x = torch.nn.functional.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)  # Output logits (softmax applied in loss)
        
        return x

# Initialize model
model = MNISTCNNModel(seed=42)
model.to(device)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 5. Training Setup

Define optimizer and loss function. PyTorch uses CrossEntropyLoss which combines softmax and cross-entropy in one efficient operation.

In [ ]:
# Setup optimizer and loss
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()  # Combines softmax + cross-entropy

# MLflow setup
mlflow.set_experiment("cnn_mnist_pytorch")

hyperparams = {
    "learning_rate": learning_rate,
    "batch_size": batch_size,
    "num_epochs": 10,  # Reduced for notebook demo
    "optimizer": "adam",
    "loss_function": "crossentropyloss",
    "num_conv_blocks": 2,
    "conv_filters_initial": 32,
    "dense_units": 128,
    "dropout_rate": 0.5,
    "random_seed": 42,
    "framework": "pytorch"
}

print("Hyperparameters:")
for key, value in hyperparams.items():
    print(f"  {key}: {value}")

## 6. Training Loop

Implement custom training loop with MLflow logging. This demonstrates PyTorch's flexibility compared to higher-level frameworks.

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
        total_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

def evaluate(model, data_loader, criterion, device):
    """Evaluate on validation/test set."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

# Train with MLflow logging
with mlflow.start_run() as run:
    mlflow.log_params(hyperparams)
    
    print(f"Training model for {hyperparams['num_epochs']} epochs...")
    start_time = time.time()
    
    for epoch in range(hyperparams['num_epochs']):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        
        # Log to MLflow
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_accuracy", train_acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_acc, step=epoch)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch + 1}: train_loss={train_loss:.4f}, val_acc={val_acc:.4f}")
    
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")
    
    run_id = run.info.run_id
    print(f"MLflow Run ID: {run_id}")

## 7. Evaluation & Visualization

Evaluate on test set and compute detailed metrics. Compare training dynamics with TensorFlow results.

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)

# Get predictions for detailed metrics
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Compute metrics
precision = precision_score(all_labels, all_predictions, average='macro')
recall = recall_score(all_labels, all_predictions, average='macro')
f1 = f1_score(all_labels, all_predictions, average='macro')

print(f"\n=== Test Set Performance ===")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")
print(f"Test F1-Score:  {f1:.4f}")

## 8. Model Persistence

Save the trained model to MLflow. PyTorch models are saved differently than TensorFlow but MLflow handles both transparently.

In [ ]:
# Save model to MLflow
mlflow.pytorch.log_model(model, artifact_path="pytorch_cnn_model")

print(f"Model saved to MLflow")
print(f"\nTo retrieve this model later:")
print(f"  mlflow.pytorch.load_model('runs:/{run_id}/pytorch_cnn_model')")

# Test loading and inference
model.eval()
sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images[:5].to(device)

with torch.no_grad():
    logits = model(sample_images)
    predicted_classes = torch.argmax(logits, dim=1)

print(f"\nSample Predictions:")
print(f"Predicted: {predicted_classes.cpu().numpy()}")
print(f"True:      {sample_labels[:5].numpy()}")